#### This notebook is used to experiment with DTGraph rules and transformations.

In [1]:
from dtgraph import Neo4jGraph, Rule, Transformation

In [2]:
hostname = "localhost"
password = "internship"
uri = f"bolt://{hostname}:7687"

graph = Neo4jGraph(uri, database="neo4j", username="neo4j", password=password)

In [3]:
from dtgraph.scenarios.movies import Movies
Movies.load(graph)

Flushed database: Deleted 342 nodes, deleted 492 relationships, completed after 291 ms.
Load scenario: Added 171 labels, created 171 nodes, set 564 properties, created 253 relationships, completed after 3209 ms.


### Node Rules 

In [4]:
generate_films = Rule('''
MATCH (m:Movie)
GENERATE
(x = (m):Film {
    title = m.title
})
''')

In [5]:
generate_humans = Rule('''
MATCH (p:Person)
GENERATE
(x = (p):Human {
    name = p.name
})
''')

In [6]:
generate_actors = Rule('''
MATCH (p:Person)-[:ACTED_IN]->(:Movie)
GENERATE
(x = (p):Actor)
''')

In [7]:
generate_directors = Rule('''
MATCH (p:Person)-[:DIRECTED]->(:Movie)
GENERATE
(x = (p):Director)
''')

In [8]:
generate_producers = Rule('''
MATCH (p:Person)-[:PRODUCED]->(:Movie)
GENERATE
(x = (p):Producer)
''')

In [9]:
generate_writers = Rule('''
MATCH (p:Person)-[:WROTE]->(:Movie)
GENERATE
(x = (p):Writer)
''')

In [10]:
generate_reviewers = Rule('''
MATCH (p:Person)-[:REVIEWED]->(:Movie)
GENERATE
(x = (p):Reviewer)
''')

### Edge Rules

In [11]:
generate_performed = Rule('''
MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
GENERATE
(x = (p):)-[():PERFORMED_IN]->(y = (m):)
''')

In [12]:
generate_created = Rule('''
MATCH (p:Person)-[:DIRECTED]->(m:Movie)
GENERATE
(x = (p):)-[():CREATED {
    role = "director"
}]->(y = (m):)
''')

In [13]:
generate_worked_on = Rule('''
MATCH (p:Person)-[r:PRODUCED|WROTE]->(m:Movie)
GENERATE
(x = (p):)-[():WORKED_ON {
    role = "PRODUCED"
}]->(y = (m):)
''')

In [14]:
generate_social = Rule('''
MATCH (a:Person)-[:FOLLOWS]->(b:Person)
GENERATE
(x = (a):)-[():FOLLOWS]->(y = (b):)
''')

### Execute Rules

In [7]:
my_transform = Transformation([
    generate_films, 
    generate_humans, 
    generate_actors, 
    # generate_directors, 
    # generate_producers, 
    # generate_writers, 
    # generate_reviewers, 
    # generate_performed, 
    # generate_created, 
    # generate_worked_on, 
    # generate_social
    ])
my_transform.apply_on(graph)

Index: Added 0 index, completed after 129 ms.
Rule: Added 76 labels, created 38 nodes, set 76 properties, created 0 relationships, completed after 650 ms.
Rule: Added 266 labels, created 133 nodes, set 266 properties, created 0 relationships, completed after 280 ms.
Rule: Added 102 labels, created 0 nodes, set 0 properties, created 0 relationships, completed after 198 ms.


1128

### Abort Transformation

In [20]:
my_transform.abort()

Index: Removed 1 index, completed after 26 ms.
Abort: Deleted 171 nodes, deleted 239 relationships, completed after 311 ms.


### Rules Not Allowed

In [ ]:
invalid_aggregation_rhs = Rule('''
MATCH (a:Person)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(b:Person)
WITH a, b, COUNT(m) AS c
RETURN a, b, c
GENERATE
(x = (a))-[():COLLAB {
    count = c
}]->(y = (b))
''')

## RHS cannot use c, no data flow
## No data flow from query phase (LHS) to construction phase (RHS)


RHS (GENERATE part)

Only understands:
node bindings (a, b)
properties (a.name)

Does NOT understand:
computed columns (c)

In [ ]:
invalid_type = Rule('''
MATCH (p:Person)-[r:ACTED_IN]->(m:Movie)
GENERATE
(x = (p))-[():WORKED_ON {
    role = type(r)
}]->(y = (m))
''')

In [ ]:
invalid_multi_rel = Rule('''
MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
GENERATE
(x = (p))-[():ACTED_IN|DIRECTED]->(y = (m))
''')

In [ ]:
invalid_undirected = Rule('''
MATCH (a:Person)--(b:Person)
GENERATE
(x = (a))-[():KNOWS]->(y = (b))
''')

In [ ]:
invalid_condition = Rule('''
MATCH (p:Person)
GENERATE
(x = (p):Actor {
    status = p.born < 1970
})
''')

In [ ]:
invalid_chain = Rule('''
MATCH (a:Person)-[:ACTED_IN]->(m:Movie)
GENERATE
(x = (a))-[():REL1]->(y = (m))-[():REL2]->(z = (a))
''')

EdgeConstructor = Node -[edge]-> Node

RHS only supports:
single 
OR single edge (between TWO nodes)